# 📊 Customer Churn Analysis — Phase 1: Data Collection

---

## Project Overview

**Goal:** Build an end-to-end customer churn analysis pipeline — identify at-risk customers, understand why they churn, and generate actionable insights.

**What this notebook does:**
- Connects to Google Drive and sets up the project folder structure
- Loads 2 real public datasets (Telco Customer Churn + Online Retail II)
- Simulates 2 realistic supporting datasets (usage logs + support tickets)
- Simulates 1 billing/payment history dataset (replacing Online Retail II for merging)
- Introduces realistic data messiness to practise cleaning
- Runs a first-pass inspection of all datasets
- Saves all raw datasets to Drive

## Datasets Used

| Dataset | Source | Rows | Description |
|---|---|---|---|
| `customers` | Telco Customer Churn (Kaggle) | ~7,043 | Core customer profile + churn label |
| `transactions` | Online Retail II (UCI) | ~400k | Used for EDA practice only — different customer universe |
| `usage` | Simulated | 3,000 | Login sessions per customer |
| `tickets` | Simulated | 1,200 | Support ticket history per customer |
| `billing` | Simulated | ~14,000 | Monthly billing and payment records per customer |

> **Note on Online Retail II:** This dataset uses integer customer IDs from UK retail shoppers — a completely different customer universe from the Telco dataset. It cannot be merged with `customers`. It is kept here for standalone EDA practice. The `billing` dataset is the correct 4th dataset for merging, as it is simulated from the same Telco customer IDs.

## Notebook Map
```
01_data_collection.ipynb   ← YOU ARE HERE
02_cleaning_merging.ipynb
03_eda_feature_engineering.ipynb
04_modelling.ipynb
```

---
## 1. Connect to Google Drive & Create Folder Structure

We mount Google Drive so all data and notebooks persist across Colab sessions. Then we create a clean folder structure:
- `data/raw/` — original, untouched files (never overwrite these)
- `data/processed/` — cleaned and merged files produced in later notebooks
- `notebooks/` — all `.ipynb` files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/customer-churn-project"

folders = [
    "data/raw",
    "data/processed",
    "notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

RAW_PATH       = os.path.join(BASE_PATH, "data/raw/")
PROCESSED_PATH = os.path.join(BASE_PATH, "data/processed/")

print("Folders ready:")
for folder in folders:
    print(f"  {BASE_PATH}/{folder}")

---
## 2. Import Libraries

We only need `pandas` and `numpy` for this phase:
- `pandas` — load, inspect, and manipulate tabular data
- `numpy` — generate random numbers for our simulated datasets

In [ ]:
import pandas as pd
import numpy as np

# Reproducibility: any code using np.random will produce the same result every run
np.random.seed(42)

print(f"pandas  version: {pd.__version__}")
print(f"numpy   version: {np.__version__}")

---
## 3. Load Real Datasets

### 3a. Telco Customer Churn (`customers`)

**Source:** [Kaggle — Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
**File:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`

This is the **core dataset** of the entire project. Each row is one customer. The `Churn` column is our target variable — the thing we want to predict.

Steps applied on load:
1. Rename `customerID` → `customer_id` for consistency
2. Encode `Churn` as binary: `Yes → 1`, `No → 0` (required for modelling later)
3. Strip whitespace from all string columns (a common source of silent bugs)

In [ ]:
churn_path = f"{RAW_PATH}WA_Fn-UseC_-Telco-Customer-Churn.csv"
customers  = pd.read_csv(churn_path)

# Rename the ID column to match all other datasets
customers = customers.rename(columns={"customerID": "customer_id"})

# Convert Churn from text to binary number (Yes=1, No=0)
customers["Churn"] = customers["Churn"].map({"Yes": 1, "No": 0})

# Strip leading/trailing whitespace from all text columns
for col in customers.select_dtypes(include="object").columns:
    customers[col] = customers[col].str.strip()

print(f"Shape: {customers.shape}")
print(f"Churn rate: {customers['Churn'].mean():.1%}")
customers.head()

### 3b. Online Retail II (`transactions`)

**Source:** [UCI ML Repository — Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii)  
**File:** `online_retail_II.xlsx`

> ⚠️ **Important:** This dataset contains UK retail transaction data with integer customer IDs (e.g. `13085`). These are **not** the same customers as in the Telco dataset. This dataset is loaded here for standalone EDA and transaction analysis practice only — it will **not** be merged with `customers`. The `billing` dataset (Section 5c) is the correct dataset to use for merging.

Steps applied on load:
1. Standardise all column names to `snake_case`
2. Drop rows with no `customer_id` (anonymous transactions are unusable for churn analysis)

In [ ]:
retail_path  = f"{RAW_PATH}online_retail_II.xlsx"
transactions = pd.read_excel(retail_path)

# Standardise column names: strip spaces, lowercase, replace spaces with underscores
transactions.columns = (
    transactions.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Drop rows where customer_id is missing — we can't attribute them to anyone
transactions = transactions[transactions["customer_id"].notna()]
transactions["customer_id"] = transactions["customer_id"].astype(int)

print(f"Shape: {transactions.shape}")
print(f"Unique customers: {transactions['customer_id'].nunique()}")
print(f"Date range: {transactions['invoicedate'].min()} → {transactions['invoicedate'].max()}")
transactions.head()

---
## 4. Introduce Realistic Data Messiness (to `customers`)

Real-world data is never clean. To practise proper cleaning techniques in the next notebook, we deliberately introduce three common types of messiness into `customers`:

| Problem | Where | Why it's realistic |
|---|---|---|
| Random missing values | `tenure`, `MonthlyCharges` | CRM systems often have incomplete records |
| Inconsistent category casing | `Contract`, `InternetService` | Data entered by different people or systems |
| Duplicate rows | Entire dataframe | ETL pipeline bugs, double imports |

> We set `np.random.seed(42)` at the top so this messiness is **reproducible** — running the notebook again produces identical results.

In [ ]:
# --- Missing values ---
# Randomly set 5% of tenure values to NaN
customers.loc[customers.sample(frac=0.05).index, "tenure"] = np.nan
# Randomly set 3% of MonthlyCharges values to NaN
customers.loc[customers.sample(frac=0.03).index, "MonthlyCharges"] = np.nan

# --- Inconsistent categories ---
# Add a trailing space to first 11 Contract values — looks identical, behaves differently
customers.loc[:10, "Contract"] = customers.loc[:10, "Contract"] + " "
# Uppercase some InternetService values — 'FIBER OPTIC' vs 'Fiber optic'
customers.loc[11:20, "InternetService"] = customers.loc[11:20, "InternetService"].str.upper()

# --- Duplicate rows ---
# Randomly sample 50 rows and append them — simulates a double-import bug
customers = pd.concat([customers, customers.sample(50)], ignore_index=True)

print(f"Shape after messiness: {customers.shape}")
print(f"Missing tenure: {customers['tenure'].isna().sum()}")
print(f"Missing MonthlyCharges: {customers['MonthlyCharges'].isna().sum()}")
print(f"\nContract unique values (note duplicates with trailing space):")
print(customers["Contract"].value_counts().to_string())
print(f"\nInternetService unique values (note UPPERCASE variants):")
print(customers["InternetService"].value_counts().to_string())

---
## 5. Create Simulated Supporting Datasets

Real telecom companies collect far more than just customer profile data. We simulate three additional tables that a real company would have, all using the same Telco customer IDs so they can be merged later.

### 5a. Usage Logs (`usage`)

Tracks each customer's app/service login activity. Features like low login frequency or short session duration are strong early-warning churn signals.

In [ ]:
# Use only the original customers (before duplicates were added)
real_customer_ids = customers.drop_duplicates(subset="customer_id")["customer_id"].values

usage = pd.DataFrame({
    # Randomly assign customer IDs (some customers will have multiple records)
    "customer_id":      np.random.choice(real_customer_ids, 3000),
    # Session duration in minutes: uniform between 5 and 300
    "session_duration": np.random.randint(5, 300, 3000),
    # Number of logins: Poisson distribution (most customers have ~5, some have more)
    "logins":           np.random.poisson(5, 3000),
    # Login date: random day within 2022
    "login_date":       pd.to_datetime("2022-01-01") +
                        pd.to_timedelta(np.random.randint(0, 365, 3000), unit="D")
})

print(f"Shape: {usage.shape}")
print(f"Unique customers in usage: {usage['customer_id'].nunique()} / {len(real_customer_ids)}")
usage.head()

### 5b. Support Tickets (`tickets`)

Tracks customer support interactions. High ticket volume, long resolution times, and frequent billing disputes are all known churn predictors. We weight billing issues at 50% because billing problems are the #1 reason customers leave.

In [ ]:
tickets = pd.DataFrame({
    "customer_id": np.random.choice(real_customer_ids, 1200),
    # Billing issues are most common (50%), followed by Technical (30%) and General (20%)
    "issue_type":  np.random.choice(
                       ["Billing", "Technical", "General"],
                       1200,
                       p=[0.5, 0.3, 0.2]
                   ),
    # Resolution time in hours: exponential distribution (most resolve quickly, some take days)
    "resolution_time_hours": np.round(np.random.exponential(scale=24, size=1200), 1),
    "ticket_date": pd.to_datetime("2022-01-01") +
                   pd.to_timedelta(np.random.randint(0, 365, 1200), unit="D")
})

print(f"Shape: {tickets.shape}")
print(f"\nIssue type distribution:")
print(tickets["issue_type"].value_counts())
tickets.head()

### 5c. Billing History (`billing`)

Tracks monthly billing records per customer. This is the **correct 4th dataset to merge** with `customers` — it shares the same Telco customer IDs and adds financially-grounded churn signals:
- `late_payment` — customers who pay late are significantly more likely to churn
- `payment_gap` — the difference between amount due and amount paid (partial payments signal financial stress)

In [ ]:
records = []

for cid in real_customer_ids:
    # Each customer has 1–3 billing records
    n_months = np.random.randint(1, 4)
    for _ in range(n_months):
        amount_due  = round(np.random.uniform(20, 120), 2)
        late        = np.random.choice([0, 1], p=[0.85, 0.15])  # 15% late payment rate
        # Late payers pay between 50–95% of the bill; on-time payers pay in full
        amount_paid = round(amount_due * np.random.uniform(0.5, 0.95), 2) if late else amount_due

        records.append({
            "customer_id":   cid,
            "billing_date":  pd.Timestamp("2022-01-01") +
                             pd.to_timedelta(np.random.randint(0, 365), unit="D"),
            "amount_due":    amount_due,
            "amount_paid":   amount_paid,
            "payment_gap":   round(amount_due - amount_paid, 2),
            "late_payment":  late
        })

billing = pd.DataFrame(records)

print(f"Shape: {billing.shape}")
print(f"Unique customers in billing: {billing['customer_id'].nunique()}")
print(f"Late payment rate: {billing['late_payment'].mean():.1%}")
billing.head()

---
## 6. Save All Raw Datasets

We save every dataset to `data/raw/` before any cleaning. This is a critical data engineering habit:
- Raw files are your **safety net** — you can always restart from here
- The next notebook (`02_cleaning_merging`) will load from raw and save its output to `data/processed/`
- Never overwrite raw files

In [ ]:
customers.to_csv(   f"{RAW_PATH}customers.csv",    index=False)
transactions.to_csv(f"{RAW_PATH}transactions.csv", index=False)
usage.to_csv(       f"{RAW_PATH}usage.csv",         index=False)
tickets.to_csv(     f"{RAW_PATH}tickets.csv",       index=False)
billing.to_csv(     f"{RAW_PATH}billing.csv",       index=False)

print("All datasets saved to data/raw/:")
for fname in ["customers.csv", "transactions.csv", "usage.csv", "tickets.csv", "billing.csv"]:
    path = f"{RAW_PATH}{fname}"
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fname:<25} {size_kb:>8.1f} KB")

---
## 7. First-Pass Inspection

Before calling any data "ready", we always run a structured inspection. This tells us:
- How many rows and columns each dataset has
- Which columns exist and what type they are
- How many values are missing per column
- What the first few rows look like

This is the foundation for the cleaning plan in notebook 02.

In [ ]:
def inspect_df(df, name):
    """
    Print a structured summary of a DataFrame:
    shape, column names and dtypes, missing value counts, and first 3 rows.
    """
    print(f"{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"Shape:   {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nColumn dtypes:")
    print(df.dtypes.to_string())
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\nMissing values:")
        print(missing.to_string())
    else:
        print(f"\nMissing values: none")
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())
    print()

In [ ]:
inspect_df(customers,    "customers  (Telco Churn — real)")
inspect_df(transactions, "transactions  (Online Retail II — real, standalone only)")
inspect_df(usage,        "usage  (simulated)")
inspect_df(tickets,      "tickets  (simulated)")
inspect_df(billing,      "billing  (simulated — merges with customers)")

---
## 8. Duplicate Check

Duplicate rows silently inflate your counts and bias your models. We check all datasets here. We expect to find duplicates in `customers` (we injected 50) and in `transactions` (common in raw retail data).

In [ ]:
print("Duplicate row counts (full row duplicates):")
for name, df in [("customers", customers), ("transactions", transactions),
                 ("usage", usage), ("tickets", tickets), ("billing", billing)]:
    n = df.duplicated().sum()
    flag = " ← needs cleaning" if n > 0 else ""
    print(f"  {name:<15} {n:>6,} duplicates{flag}")

---
## 9. Key Variable Checks

### 9a. Churn Rate

The churn rate is the single most important number in this project. It tells us how imbalanced our target variable is, which directly affects which models we choose and how we evaluate them.

In [ ]:
churn_counts = customers["Churn"].value_counts()
churn_rate   = customers["Churn"].mean()

print(f"Churn distribution:")
print(f"  Did NOT churn (0): {churn_counts[0]:,}  ({1 - churn_rate:.1%})")
print(f"  Churned       (1): {churn_counts[1]:,}  ({churn_rate:.1%})")
print(f"")
print(f"Class imbalance ratio: {churn_counts[0] / churn_counts[1]:.1f}:1")
print(f"")
print("Implication: our dataset is moderately imbalanced.")
print("In notebook 04 we will use techniques like SMOTE or class_weight to handle this.")

### 9b. Category Consistency Check

Here we verify the messiness we introduced is actually present, and understand exactly what we need to fix in notebook 02.

In [ ]:
print("Contract value counts (should see trailing-space duplicates):")
print(customers["Contract"].value_counts().to_string())

print("\nInternetService value counts (should see UPPERCASE variants):")
print(customers["InternetService"].value_counts().to_string())

print("\nTop 5 countries in transactions:")
print(transactions["country"].value_counts().head().to_string())

---
## 10. Phase 1 Summary

Here we record a concise summary of everything we found. This is good notebook hygiene — a reader (or future you) can understand the state of the data without re-running all cells.

In [ ]:
print("PHASE 1 COMPLETE — DATA COLLECTION SUMMARY")
print("=" * 55)
print()
print("Datasets loaded and saved to data/raw/:")
print(f"  customers    : {customers.shape[0]:,} rows, {customers.shape[1]} cols | churn rate: {customers['Churn'].mean():.1%}")
print(f"  transactions : {transactions.shape[0]:,} rows, {transactions.shape[1]} cols | standalone only")
print(f"  usage        : {usage.shape[0]:,} rows, {usage.shape[1]} cols")
print(f"  tickets      : {tickets.shape[0]:,} rows, {tickets.shape[1]} cols")
print(f"  billing      : {billing.shape[0]:,} rows, {billing.shape[1]} cols")
print()
print("Known issues to fix in notebook 02:")
print(f"  customers : {customers.duplicated().sum()} duplicate rows")
print(f"  customers : {customers['tenure'].isna().sum()} missing tenure values")
print(f"  customers : {customers['MonthlyCharges'].isna().sum()} missing MonthlyCharges values")
print(f"  customers : inconsistent casing in Contract + InternetService columns")
print(f"  customers : mixed column naming (CamelCase) — needs snake_case")
print(f"  transactions: {transactions.duplicated().sum():,} duplicate rows")
print()
print("Next: 02_cleaning_merging.ipynb")